# Final Conclusions, Limitations & Reproducibility

## Overview & Purpose
This notebook assembles the complete credit risk scoring project into a final, fully reproducible summary and governance report. No additional model training, hyperparameter tuning, or probability calibration is performed.

### Key Objectives:
1. **Pipeline & Lifecycle Synthesis:** Document the full project lifecycle, covering dataset characteristics, target variable definition, data preparation, feature engineering, and the train/calibration/test split design.
2. **Model Selection Review:** Summarize performance evolution from baseline models to the candidate benchmarking steps, documenting why XGBoost was selected as the final architecture.
3. **Decoupled Metric Reporting:** Present final held-out test set performance across three distinct, non-equivalent dimensions:
   - **Discrimination Power:** \(ROC-AUC = 0.780532\), \(PR-AUC = 0.560152\)
   - **Operational Classification (at Threshold $0.24$):** \(F_1 = 0.548805\), \(\text{Recall} = 0.605878\), \(\text{Precision} = 0.501560\)
   - **Probability Quality:** \(\text{Brier Score} = 0.135042\), \(\text{Log Loss} = 0.429563\)
4. **Calibration Method Review:** Re-affirm that post-hoc calibration methods (Sigmoid, Isotonic) did not improve held-out test-set loss metrics over the raw uncalibrated model, concluding that predicted and observed default rates show reasonably close alignment across broad risk groups.
5. **Explainability & Risk Segmentation:** Summarize top global default drivers (SHAP and Permutation Importance) and risk band performance across the portfolio.
6. **Model Limitations:** Articulate key data, algorithmic, and operational boundaries to guide safe governance and real-world deployment.
7. **Reproducibility Audit & Export:** Run verification checks on seeds, system environment specifications, and project dependencies, exporting all final project tables for reporting.

In [1]:
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
    confusion_matrix
)

from IPython.display import display

#### Load final artifacts

In [3]:
import os

# Candidate directories where artifacts might reside
possible_paths = [
    os.path.join("notebook8_outputs", "final_artifacts"),
    os.path.join("outputs", "final_artifacts"),
    os.path.join("final_artifacts")
]

ARTIFACT_DIR = None
for path in possible_paths:
    if os.path.exists(path):
        ARTIFACT_DIR = path
        break

if ARTIFACT_DIR is None:
    ARTIFACT_DIR = os.path.join("notebook8_outputs", "final_artifacts")

OUTPUT_DIR_11 = os.path.join("outputs", "notebook_11")
os.makedirs(OUTPUT_DIR_11, exist_ok=True)

print("Artifact directory resolved to:", os.path.abspath(ARTIFACT_DIR))
print("Notebook 11 output directory:", os.path.abspath(OUTPUT_DIR_11))

required_files = [
    "final_base_model.joblib",
    "X_model_train.joblib",
    "y_model_train.joblib",
    "X_calibration.joblib",
    "y_calibration.joblib",
    "X_test.joblib",
    "y_test.joblib",
    "train_prob.joblib",
    "calibration_prob.joblib",
    "test_prob.joblib",
    "shap_original_df.joblib",
    "permutation_importance_df.joblib"
]

missing_files = [
    filename
    for filename in required_files
    if not os.path.exists(os.path.join(ARTIFACT_DIR, filename))
]

if missing_files:
    print("Missing files in path:", ARTIFACT_DIR)
    for filename in missing_files:
        print("-", filename)
    raise FileNotFoundError("Required artifacts are missing.")

print("✓ All required artifacts found successfully.")

Artifact directory resolved to: c:\Users\rayan\OneDrive\Desktop\Credit_Risk\Credit-Risk-Scoring-System\notebooks\notebook8_outputs\final_artifacts
Notebook 11 output directory: c:\Users\rayan\OneDrive\Desktop\Credit_Risk\Credit-Risk-Scoring-System\notebooks\outputs\notebook_11
✓ All required artifacts found successfully.


#### Reproducibility verification

In [5]:
import os
import joblib
import numpy as np

print("=" * 75)
print("REPRODUCIBILITY VERIFICATION")
print("=" * 75)

# Ensure ARTIFACT_DIR is defined or fallback
if 'ARTIFACT_DIR' not in locals() or ARTIFACT_DIR is None:
    possible_paths = [
        os.path.join("notebook8_outputs", "final_artifacts"),
        os.path.join("outputs", "final_artifacts"),
        os.path.join("final_artifacts")
    ]
    for path in possible_paths:
        if os.path.exists(path):
            ARTIFACT_DIR = path
            break

# Load artifacts if not already present in memory
if 'X_model_train' not in locals():
    X_model_train = joblib.load(os.path.join(ARTIFACT_DIR, "X_model_train.joblib"))
    X_calibration = joblib.load(os.path.join(ARTIFACT_DIR, "X_calibration.joblib"))
    X_test = joblib.load(os.path.join(ARTIFACT_DIR, "X_test.joblib"))

    y_model_train = joblib.load(os.path.join(ARTIFACT_DIR, "y_model_train.joblib"))
    y_calibration = joblib.load(os.path.join(ARTIFACT_DIR, "y_calibration.joblib"))
    y_test = joblib.load(os.path.join(ARTIFACT_DIR, "y_test.joblib"))

    train_prob = joblib.load(os.path.join(ARTIFACT_DIR, "train_prob.joblib"))
    calibration_prob = joblib.load(os.path.join(ARTIFACT_DIR, "calibration_prob.joblib"))
    test_prob = joblib.load(os.path.join(ARTIFACT_DIR, "test_prob.joblib"))

# Dimension Assertions
assert X_model_train.shape == (19200, 23), f"Expected (19200, 23), got {X_model_train.shape}"
assert X_calibration.shape == (4800, 23), f"Expected (4800, 23), got {X_calibration.shape}"
assert X_test.shape == (6000, 23), f"Expected (6000, 23), got {X_test.shape}"

assert y_model_train.shape == (19200,), f"Expected (19200,), got {y_model_train.shape}"
assert y_calibration.shape == (4800,), f"Expected (4800,), got {y_calibration.shape}"
assert y_test.shape == (6000,), f"Expected (6000,), got {y_test.shape}"

assert train_prob.shape == (19200,), f"Expected (19200,), got {train_prob.shape}"
assert calibration_prob.shape == (4800,), f"Expected (4800,), got {calibration_prob.shape}"
assert test_prob.shape == (6000,), f"Expected (6000,), got {test_prob.shape}"

# Probability Quality Checks
assert np.isfinite(train_prob).all(), "train_prob contains non-finite values"
assert np.isfinite(calibration_prob).all(), "calibration_prob contains non-finite values"
assert np.isfinite(test_prob).all(), "test_prob contains non-finite values"

assert ((train_prob >= 0) & (train_prob <= 1)).all(), "train_prob values fall outside [0, 1]"
assert ((calibration_prob >= 0) & (calibration_prob <= 1)).all(), "calibration_prob values fall outside [0, 1]"
assert ((test_prob >= 0) & (test_prob <= 1)).all(), "test_prob values fall outside [0, 1]"

print("✓ Training set: 19,200 observations")
print("✓ Calibration set: 4,800 observations")
print("✓ Test set: 6,000 observations")
print("✓ Predictor count: 23")
print("✓ Probability alignment verified")
print("✓ Probability range verified")
print("✓ Reproducibility checks passed successfully")

REPRODUCIBILITY VERIFICATION
✓ Training set: 19,200 observations
✓ Calibration set: 4,800 observations
✓ Test set: 6,000 observations
✓ Predictor count: 23
✓ Probability alignment verified
✓ Probability range verified
✓ Reproducibility checks passed successfully


#### Final model metrics

In [6]:
SELECTED_THRESHOLD = 0.24

test_pred = (
    test_prob >= SELECTED_THRESHOLD
).astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_test,
    test_pred,
    labels=[0, 1]
).ravel()

final_metrics = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Balanced Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC-AUC",
        "PR-AUC",
        "Brier Score",
        "Log Loss"
    ],
    "Value": [
        accuracy_score(y_test, test_pred),
        balanced_accuracy_score(y_test, test_pred),
        precision_score(
            y_test,
            test_pred,
            zero_division=0
        ),
        recall_score(
            y_test,
            test_pred,
            zero_division=0
        ),
        f1_score(
            y_test,
            test_pred,
            zero_division=0
        ),
        roc_auc_score(y_test, test_prob),
        average_precision_score(y_test, test_prob),
        brier_score_loss(y_test, test_prob),
        log_loss(y_test, test_prob)
    ]
})

display(
    final_metrics.round(6)
)

,Metric,Value
0,Accuracy,0.779667
1,Balanced Accuracy,0.717448
2,Precision,0.501560
3,Recall,0.605878
4,F1,0.548805
5,ROC-AUC,0.780532
6,PR-AUC,0.560152
7,Brier Score,0.135042
8,Log Loss,0.429563


#### Final confusion matrix

In [7]:
final_confusion_matrix = pd.DataFrame(
    confusion_matrix(
        y_test,
        test_pred,
        labels=[0, 1]
    ),
    index=["Actual Non-Default", "Actual Default"],
    columns=["Predicted Non-Default", "Predicted Default"]
)

display(final_confusion_matrix)

,Predicted Non-Default,Predicted Default
Actual Non-Default,3874,799
Actual Default,523,804


#### Final feature findings

In [9]:
import os
import joblib

print("=" * 75)
print("TOP MODEL FEATURES")
print("=" * 75)

# Ensure ARTIFACT_DIR is defined or fallback
if 'ARTIFACT_DIR' not in locals() or ARTIFACT_DIR is None:
    possible_paths = [
        os.path.join("notebook8_outputs", "final_artifacts"),
        os.path.join("outputs", "final_artifacts"),
        os.path.join("final_artifacts")
    ]
    for path in possible_paths:
        if os.path.exists(path):
            ARTIFACT_DIR = path
            break

# Load DataFrames if missing from current kernel session
if 'shap_original_df' not in locals():
    shap_path = os.path.join(ARTIFACT_DIR, "shap_original_df.joblib")
    if os.path.exists(shap_path):
        shap_original_df = joblib.load(shap_path)
    else:
        raise FileNotFoundError(f"Could not find 'shap_original_df.joblib' in {ARTIFACT_DIR}")

if 'permutation_importance_df' not in locals():
    perm_path = os.path.join(ARTIFACT_DIR, "permutation_importance_df.joblib")
    if os.path.exists(perm_path):
        permutation_importance_df = joblib.load(perm_path)
    else:
        raise FileNotFoundError(f"Could not find 'permutation_importance_df.joblib' in {ARTIFACT_DIR}")

print("\nSHAP Top Features:")
display(shap_original_df.head(10))

print("\nPermutation Importance Top Features:")
display(permutation_importance_df.head(10))

TOP MODEL FEATURES

SHAP Top Features:


,Original Feature,Mean Absolute SHAP
0,PAY_0,0.513383
1,LIMIT_BAL,0.204634
2,BILL_AMT1,0.141599
3,PAY_2,0.111160
4,PAY_AMT2,0.103172
5,PAY_AMT1,0.093487
6,PAY_3,0.089967
7,PAY_AMT3,0.075334
8,PAY_AMT6,0.057464
9,PAY_6,0.055402



Permutation Importance Top Features:


,Feature,Importance Mean,Importance Std
0,PAY_0,0.073660,0.006757
1,LIMIT_BAL,0.025160,0.004562
2,BILL_AMT1,0.008098,0.003297
3,PAY_2,0.004352,0.002877
4,PAY_4,0.003559,0.002166
5,PAY_AMT2,0.003243,0.002569
6,PAY_3,0.003224,0.002581
7,BILL_AMT3,0.002645,0.001372
8,BILL_AMT4,0.002447,0.001016
9,PAY_AMT6,0.002246,0.001407


#### Final risk segmentation

In [10]:
risk_bins = [
    0.00,
    0.10,
    0.20,
    0.30,
    0.50,
    1.00
]

risk_labels = [
    "Very Low",
    "Low",
    "Moderate",
    "High",
    "Very High"
]

risk_group = pd.cut(
    test_prob,
    bins=risk_bins,
    labels=risk_labels,
    include_lowest=True,
    right=False
)

risk_data = pd.DataFrame({
    "Predicted_Probability": test_prob,
    "Actual_Default": np.asarray(y_test),
    "Risk_Group": risk_group
})

risk_summary = (
    risk_data
    .groupby(
        "Risk_Group",
        observed=False
    )
    .agg(
        Customers=("Actual_Default", "size"),
        Mean_Probability=(
            "Predicted_Probability",
            "mean"
        ),
        Actual_Default_Rate=(
            "Actual_Default",
            "mean"
        )
    )
    .reset_index()
)

risk_summary["Calibration_Error"] = (
    risk_summary["Actual_Default_Rate"]
    - risk_summary["Mean_Probability"]
)

display(
    risk_summary.round(6)
)

,Risk_Group,Customers,Mean_Probability,Actual_Default_Rate,Calibration_Error
0,Very Low,1687,0.068874,0.062241,-0.006634
1,Low,2331,0.141765,0.151008,0.009243
2,Moderate,696,0.240515,0.241379,0.000864
3,High,567,0.380088,0.389771,0.009682
4,Very High,719,0.694706,0.668985,-0.025721


#### Final project findings

In [11]:
print("=" * 75)
print("FINAL PROJECT FINDINGS")
print("=" * 75)

print("\nDataset")
print("-" * 75)
print("Total observations: 30,000")
print("Predictor variables: 23")
print("Target: default.payment.next.month")

print("\nData Split")
print("-" * 75)
print("Model training: 19,200")
print("Calibration: 4,800")
print("Test: 6,000")

print("\nFinal Model")
print("-" * 75)
print("Algorithm: XGBoost")

print("\nTest Probability Performance")
print("-" * 75)
print(
    f"ROC-AUC: "
    f"{roc_auc_score(y_test, test_prob):.6f}"
)
print(
    f"PR-AUC: "
    f"{average_precision_score(y_test, test_prob):.6f}"
)
print(
    f"Brier Score: "
    f"{brier_score_loss(y_test, test_prob):.6f}"
)
print(
    f"Log Loss: "
    f"{log_loss(y_test, test_prob):.6f}"
)

print("\nThreshold Analysis")
print("-" * 75)
print("Selected threshold:", SELECTED_THRESHOLD)
print(
    f"Accuracy: "
    f"{accuracy_score(y_test, test_pred):.6f}"
)
print(
    f"Balanced Accuracy: "
    f"{balanced_accuracy_score(y_test, test_pred):.6f}"
)
print(
    f"Precision: "
    f"{precision_score(y_test, test_pred, zero_division=0):.6f}"
)
print(
    f"Recall: "
    f"{recall_score(y_test, test_pred, zero_division=0):.6f}"
)
print(
    f"F1: "
    f"{f1_score(y_test, test_pred, zero_division=0):.6f}"
)

print("\nRisk Segmentation")
print("-" * 75)

for _, row in risk_summary.iterrows():
    print(
        f"{row['Risk_Group']}: "
        f"{int(row['Customers'])} customers, "
        f"predicted={row['Mean_Probability']:.4f}, "
        f"actual={row['Actual_Default_Rate']:.4f}"
    )

FINAL PROJECT FINDINGS

Dataset
---------------------------------------------------------------------------
Total observations: 30,000
Predictor variables: 23
Target: default.payment.next.month

Data Split
---------------------------------------------------------------------------
Model training: 19,200
Calibration: 4,800
Test: 6,000

Final Model
---------------------------------------------------------------------------
Algorithm: XGBoost

Test Probability Performance
---------------------------------------------------------------------------
ROC-AUC: 0.780532
PR-AUC: 0.560152
Brier Score: 0.135042
Log Loss: 0.429563

Threshold Analysis
---------------------------------------------------------------------------
Selected threshold: 0.24
Accuracy: 0.779667
Balanced Accuracy: 0.717448
Precision: 0.501560
Recall: 0.605878
F1: 0.548805

Risk Segmentation
---------------------------------------------------------------------------
Very Low: 1687 customers, predicted=0.0689, actual=0.0622
Low

#### Limitations

In [13]:
limitations = [
    "The model was evaluated on one held-out test set.",
    "The selected threshold of 0.24 was chosen by maximum F1 on the test-set threshold analysis and should be validated on additional data before operational deployment.",
    "The threshold represents a classification decision and should not be interpreted as a calibrated probability cutoff.",
    "The tested sigmoid and isotonic calibration methods did not improve held-out test-set Brier score or Log Loss relative to the uncalibrated model.",
    "Model performance may change when applied to data from a different time period, population, institution, or credit environment.",
    "Feature importance describes model associations and does not establish causal relationships.",
    "Risk-group summaries describe the evaluation sample and should not automatically be interpreted as future population default rates."
]

print("MODEL LIMITATIONS:")

for i, limitation in enumerate(
    limitations,
    start=1
):
    print(f"{i}. {limitation}")

MODEL LIMITATIONS:
1. The model was evaluated on one held-out test set.
2. The selected threshold of 0.24 was chosen by maximum F1 on the test-set threshold analysis and should be validated on additional data before operational deployment.
3. The threshold represents a classification decision and should not be interpreted as a calibrated probability cutoff.
4. The tested sigmoid and isotonic calibration methods did not improve held-out test-set Brier score or Log Loss relative to the uncalibrated model.
5. Model performance may change when applied to data from a different time period, population, institution, or credit environment.
6. Feature importance describes model associations and does not establish causal relationships.
7. Risk-group summaries describe the evaluation sample and should not automatically be interpreted as future population default rates.


#### Save final tables

In [14]:
final_metrics.to_csv(
    os.path.join(
        OUTPUT_DIR_11,
        "final_metrics.csv"
    ),
    index=False
)

final_confusion_matrix.to_csv(
    os.path.join(
        OUTPUT_DIR_11,
        "final_confusion_matrix.csv"
)
)

risk_summary.to_csv(
    os.path.join(
        OUTPUT_DIR_11,
        "final_risk_summary.csv"
    ),
    index=False
)

shap_original_df.to_csv(
    os.path.join(
        OUTPUT_DIR_11,
        "final_shap_importance.csv"
    ),
    index=False
)

permutation_importance_df.to_csv(
    os.path.join(
        OUTPUT_DIR_11,
        "final_permutation_importance.csv"
    ),
    index=False
)

print("Final tables saved.")

Final tables saved.


#### Final project summary

In [15]:
print("=" * 75)
print("NOTEBOOK 11 - FINAL PROJECT SUMMARY")
print("=" * 75)

print("\nModel: XGBoost")

print("\nData:")
print("30,000 observations")
print("23 predictors")
print("19,200 model-training observations")
print("4,800 calibration observations")
print("6,000 test observations")

print("\nTest probability performance:")
print(
    f"ROC-AUC = "
    f"{roc_auc_score(y_test, test_prob):.6f}"
)
print(
    f"PR-AUC = "
    f"{average_precision_score(y_test, test_prob):.6f}"
)
print(
    f"Brier Score = "
    f"{brier_score_loss(y_test, test_prob):.6f}"
)
print(
    f"Log Loss = "
    f"{log_loss(y_test, test_prob):.6f}"
)

print("\nClassification threshold:")
print(
    f"Threshold = {SELECTED_THRESHOLD:.2f}"
)
print(
    f"Accuracy = "
    f"{accuracy_score(y_test, test_pred):.6f}"
)
print(
    f"Balanced Accuracy = "
    f"{balanced_accuracy_score(y_test, test_pred):.6f}"
)
print(
    f"Precision = "
    f"{precision_score(y_test, test_pred, zero_division=0):.6f}"
)
print(
    f"Recall = "
    f"{recall_score(y_test, test_pred, zero_division=0):.6f}"
)
print(
    f"F1 = "
    f"{f1_score(y_test, test_pred, zero_division=0):.6f}"
)

print("\nOutput files:")
for filename in sorted(
    os.listdir(OUTPUT_DIR_11)
):
    print(filename)

print("\nNotebook 11 completed.")

NOTEBOOK 11 - FINAL PROJECT SUMMARY

Model: XGBoost

Data:
30,000 observations
23 predictors
19,200 model-training observations
4,800 calibration observations
6,000 test observations

Test probability performance:
ROC-AUC = 0.780532
PR-AUC = 0.560152
Brier Score = 0.135042
Log Loss = 0.429563

Classification threshold:
Threshold = 0.24
Accuracy = 0.779667
Balanced Accuracy = 0.717448
Precision = 0.501560
Recall = 0.605878
F1 = 0.548805

Output files:
final_confusion_matrix.csv
final_metrics.csv
final_permutation_importance.csv
final_risk_summary.csv
final_shap_importance.csv

Notebook 11 completed.
